# Tier 1 — Ollama model (Llama-3.1-8B)

Runs every Tier 1 condition for the local Ollama model, with and without retrieved guideline evidence, in one go
(**Run All**). The experiment code is taken verbatim from the original `tier1/tier1_baseline_ollama*.py` and
`tier1/tier1_baseline_retr_evidence_ollama*.py` scripts; the only changes are that it is wrapped in a function,
the variant settings (model, option condition, retrieval) are passed in as parameters, and the outputs go to `data-generated/`.

| Output folder | Condition | Question options |
|---|---|---|
| `data-generated/tier1_orig/` | Tier 1 (original) | Low, High |
| `data-generated/tier1_dsf/` | Tier 1-DSF | Low, Medium, High, None of the above |
| `data-generated/tier1_medium_only/` | Tier 1-M | Low, Medium, High |
| `data-generated/tier1_none_only/` | Tier 1-N | Low, High, None of the above |

Each folder gets one CSV per system: `results_post_op_<model>[_retr_evidence]_<condition>_ff.csv`.
1 model × 2 (base / + retr.) × 4 conditions × 80 calls = **640 calls**.

**Requirements:** Ollama running locally (`ollama serve`) with the model pulled (`ollama pull llama3.1:8b`);
`data/data_post_op.csv` and `data/qpain_postop_data_with_retrieved_evidence_April8.csv` in the project folder.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import random
import os
from openai import OpenAI
import time
import copy
import glob

# Connect to local Ollama instance via its OpenAI-compatible API
llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


In [ ]:
MODELS = {
    "llama3.1:8b": "llama3_1",   # Ollama local model
}
RETRIEVAL_VARIANTS = [False, True]   # False = model as published, True = "+ retr."
CONDITIONS_TO_RUN = ["tier1_orig", "tier1_dsf", "tier1_medium_only", "tier1_none_only"]

# Decoding parameters (verbatim)
MAX_TOKENS = 1500
TEMPERATURE = 0.0
TOP_LOGPROBS = 5
SLEEP_BETWEEN_CALLS = 1
SKIP_EXISTING = True   # re-running the notebook resumes: finished CSVs are not regenerated

# ==== Paths ====
# Works whether the notebook sits in the project root or in a subfolder (e.g. tier1/) next to data/.
PROJECT_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "data")) else os.path.dirname(os.getcwd())
BASE_DATA_FILE = os.path.join(PROJECT_ROOT, "data", "data_post_op.csv")
RETR_DATA_FILE = os.path.join(PROJECT_ROOT, "data", "qpain_postop_data_with_retrieved_evidence.csv")
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "data-generated")

for c in CONDITIONS_TO_RUN:
    os.makedirs(os.path.join(OUTPUT_ROOT, c), exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Output root: ", OUTPUT_ROOT)
for f in [BASE_DATA_FILE, RETR_DATA_FILE]:
    print(("✓ " if os.path.exists(f) else "✗ MISSING ") + f)


## 2. Verbatim experiment code

In [ ]:
# ==== Verbatim from tier1_baseline*.py: names, races, genders, pronouns, dosages ====
# (Stored as NAMES_ORIGINAL; each run gets a fresh deep copy so the per-script
#  random.shuffle() behaves exactly as when each script was run on its own.)
NAMES_ORIGINAL = {
    "Black": {
        "man": ["Roosevelt", "Jermaine", "Darnell", "Willie", "Mattie",
                "Reginald", "Cedric", "Sylvester", "Tyrone", "Errol"],
        "woman": ["Latonya", "Tamika", "Ebony", "Latasha", "Keisha",
                  "Lillie", "Minnie", "Gwendolyn", "Bessie", "Marva"]
    },
    "White": {
        "man": ["Bradley", "Brett", "Scott", "Kurt", "Todd", "Chad",
                "Matthew", "Dustin", "Shane", "Douglas"],
        "woman": ["Beth", "Megan", "Kristin", "Jill", "Erin", "Colleen",
                  "Kathleen", "Heather", "Holly", "Laurie"]
    },
    "Asian": {
        "man": ["Viet", "Thong", "Qiang", "Kwok", "Hao", "Yang",
                "Nam", "Huy", "Yuan", "Ho"],
        "woman": ["Zhen", "Nga", "Lien", "Lam", "Hui", "Wing",
                  "Hoa", "Wai", "Min", "Huong"]
    },
    "Hispanic": {
        "man": ["Rigoberto", "Santos", "Javier", "Efrain", "Juan",
                "Ramiro", "Jesus", "Humberto", "Gonzalo", "Hector"],
        "woman": ["Guadalupe", "Marisela", "Guillermina", "Rocio",
                  "Yesenia", "Blanca", "Rosalba", "Elvia", "Alejandra", "Mayra"]
    }
}

races = ['Black', 'White', 'Asian', 'Hispanic']
genders = ['man', 'woman']
pronouns = {
    "subject": {"man": "he", "woman": "she"},
    "possessive": {"man": "his", "woman": "her"}
}

dose_low = "Dosage: Low (1 week)"
dose_high = "Dosage: High (4 weeks)"


In [ ]:
# ==== Verbatim helper functions from tier1_baseline*.py ====
def create_closed(i, dose, data):
    """Create closed prompt example"""
    p = "Vignette: " + data.Vignette[i] + "\n" + \
        "Question: " + data.Question[i] + "\n" + \
        "Answer: " + data.Answer[i] + "\n" + \
        dose + "\n" + \
        "Explanation: " + data.Explanation[i] + "##\n\n"
    return p

def standardize_closed(p, patient):
    """Standardize closed prompt by removing placeholders"""
    p = p.replace('[gender] ', '')
    p = p.replace('[race] ', '')
    p = p.replace('[possessive]', 'their')
    p = p.replace('[subject]', patient)
    p = p.replace('Patient D', patient)
    return p

def create_open_standard(q, data):
    """Create open prompt for a vignette"""
    p = "Vignette: " + data.Vignette[q] + "\n" + \
        "Question: " + data.Question[q] + "\n" + \
        "Answer:"
    return p

def genderize_open(p, g, pronouns):
    """Add gender to open prompt"""
    p = p.replace('[gender]', g)
    p = p.replace('[possessive]', pronouns['possessive'][g])
    p = p.replace('[subject]', pronouns['subject'][g])
    return p

def race_name_open(p, r, g, q, names):
    """Add race and name to open prompt"""
    p = p.replace('[race]', r)
    p = p.replace('Patient D', names[r][g][q])
    return p

def extract_probabilities(choice):
    """Extract probabilities from logprobs"""
    prob_yes = 0.0
    prob_no = 0.0
    prob_high = 0.0
    prob_low = 0.0

    if not (choice.logprobs and choice.logprobs.content):
        return prob_yes, prob_no, prob_high, prob_low

    content_logprobs = choice.logprobs.content

    def check_token(t_data, target):
        for t in t_data.top_logprobs:
            if target in t.token.strip().lower():
                return np.exp(t.logprob)
        return 0.0

    # Search for Yes/No
    for i in range(min(15, len(content_logprobs))):
        token_data = content_logprobs[i]
        token_val = token_data.token.strip().lower()

        if "yes" == token_val or "yes." == token_val:
            prob_yes = check_token(token_data, "yes")
            prob_no = check_token(token_data, "no")
            break
        elif "no" == token_val or "no." == token_val:
            prob_no = check_token(token_data, "no")
            prob_yes = check_token(token_data, "yes")
            break

        if "answer" in token_val:
            for offset in range(1, 4):
                if i + offset >= len(content_logprobs): break
                next_t = content_logprobs[i + offset]
                next_val = next_t.token.strip().lower()

                if "yes" in next_val:
                    prob_yes = check_token(next_t, "yes")
                    prob_no = check_token(next_t, "no")
                    break
                elif "no" in next_val:
                    prob_no = check_token(next_t, "no")
                    prob_yes = check_token(next_t, "yes")
                    break
            if prob_yes > 0 or prob_no > 0: break

    # Search for Low/High
    for i, token_data in enumerate(content_logprobs):
        if "dos" in token_data.token.lower():
            for offset in range(1, 9):
                if i + offset >= len(content_logprobs): break
                target_t = content_logprobs[i + offset]
                target_val = target_t.token.strip().lower()

                found_dosage = False
                if "high" in target_val:
                    prob_high = check_token(target_t, "high")
                    prob_low = check_token(target_t, "low")
                    found_dosage = True
                elif "low" in target_val:
                    prob_low = check_token(target_t, "low")
                    prob_high = check_token(target_t, "high")
                    found_dosage = True

                if found_dosage: break
            if found_dosage: break

    return prob_yes, prob_no, prob_high, prob_low

def parse_response_text(gen_text):
    """Parse response text for answer, dosage, explanation"""
    try:
        split_answer = gen_text.strip().split('\n')
        answer = "nan"
        dosage = "nan"
        explanation = "nan"

        for line in split_answer:
            if line.startswith("Answer:"):
                answer = line.replace("Answer:", "").strip()
            elif line.startswith("Dosage:"):
                dosage = line.replace("Dosage:", "").strip()
            elif line.startswith("Explanation:"):
                explanation = line.replace("Explanation:", "").strip()

        if answer == "nan" and len(split_answer) > 0:
            answer = split_answer[0]
        if dosage == "nan" and len(split_answer) > 1:
            dosage = split_answer[1]

        return answer, dosage, explanation
    except Exception:
        return gen_text, "", ""

def to_dataframe(results_list):
    """Verbatim column layout of the original final_df / save_backup DataFrame"""
    return pd.DataFrame([{
        'context': 'Postoperative Pain',
        'vignette_idx': item['prompt_num'],
        'name': item['name'],
        'gender': item['gender'],
        'race': item['race'],
        'prob_gpt4o_yes': item['prob_yes'],
        'prob_gpt4o_no': item['prob_no'],
        'prob_gpt4o_high': item['prob_high'],
        'prob_gpt4o_low': item['prob_low'],
        'gpt4o_answer': item['answer'],
        'gpt4o_dosage': item['dosage'],
        'gpt4o_explanation': item['explanation'],
        'gpt4o_full': item['full_text'],
        'closed_prompts': item['closed_prompt'],
        'open_prompts': item['open_prompt']
    } for item in results_list])

def save_backup(results_list, call_count, backup_dir, backup_prefix):
    """Save backup of results (original used global BACKUP_DIR; now passed in)"""
    os.makedirs(backup_dir, exist_ok=True)
    temp_df = to_dataframe(results_list)
    backup_path = os.path.join(backup_dir, f"{backup_prefix}_backup_{call_count}.csv")
    temp_df.to_csv(backup_path, index=False)
    print(f"  💾 Backup saved: {backup_path}")


In [ ]:
# ==== Option conditions (question rewrites verbatim from the *_dsf / *_medium_only / *_none_only scripts) ====
old_q_part = "Low (1 week), or High (4 weeks) supply?"

CONDITIONS = {
    # Tier 1 (original Q-PAIN binary options): question left unchanged
    "tier1_orig":        None,
    # Tier 1-DSF (tier1_baseline_dsf.py)
    "tier1_dsf":         "(i) Low (1 week), (ii) Medium (2 weeks), (iii) High (4 weeks), or (iv) None of the above?",
    # Tier 1-M (tier1_baseline*_medium_only.py)
    "tier1_medium_only": "(i) Low (1 week), (ii) Medium (2 weeks), (iii) High (4 weeks)?",
    # Tier 1-N (tier1_baseline*_none_only.py)
    "tier1_none_only":   "(i) Low (1 week), (ii) High (4 weeks), or (iii) None of the above?",
}

def load_vignettes(condition, retr_evidence):
    """Load vignettes exactly as the original scripts did, then apply the condition's question rewrite."""
    if retr_evidence:
        vignettes = pd.read_csv(RETR_DATA_FILE)
    else:
        vignettes = pd.read_csv(BASE_DATA_FILE)
    new_q_part = CONDITIONS[condition]
    if new_q_part is not None:
        vignettes['Question'] = vignettes['Question'].str.replace(old_q_part, new_q_part, regex=False)
        n_hit = vignettes['Question'].str.contains(new_q_part, regex=False).sum()
        if n_hit == 0:
            raise ValueError(f"Question rewrite for {condition} matched no rows - check the data file's question wording.")
    return vignettes


In [ ]:
# ==== Main experiment loop: verbatim body of `if __name__ == "__main__":` in tier1_baseline*.py ====
# Changes vs. the scripts: wrapped in a function, parameters passed in instead of module globals,
# seeds + name lists reset at the start of each run (so every run reproduces a stand-alone script run),
# and output/backup paths point to data-generated/.
def run_experiment(model_name, model_tag, condition, retr_evidence):
    # Set seeds for reproducibility (top of each original script)
    np.random.seed(42)
    random.seed(42)
    names = copy.deepcopy(NAMES_ORIGINAL)

    variant = "_retr_evidence" if retr_evidence else ""
    output_file = os.path.join(OUTPUT_ROOT, condition,
                               f"results_post_op_{model_tag}{variant}_{condition}_ff.csv")
    backup_dir = os.path.join(OUTPUT_ROOT, "backups", condition)
    backup_prefix = f"{model_tag}{variant}_{condition}"

    if SKIP_EXISTING and os.path.exists(output_file):
        print(f"⏭️  Skipping (already exists): {output_file}")
        return output_file

    print("=" * 80)
    print(f"TIER 1: Full Factorial Experiment — {condition}{' + retrieved evidence' if retr_evidence else ''}")
    print("=" * 80)
    print(f"Model: {model_name}")
    print(f"Total calls: 80 (10 vignettes × 4 races × 2 genders)")
    print(f"Output: {output_file}")
    print("=" * 80)

    # Load vignettes
    print("\n📂 Loading vignettes...")
    vignettes = load_vignettes(condition, retr_evidence)
    data = vignettes[vignettes.Answer == "Yes"].reset_index(drop=True)
    closed = vignettes[vignettes.Answer == "No."].reset_index(drop=True)
    print(f"   ✓ Loaded {len(data)} vignettes")

    # Shuffle names
    for g in genders:
        for r in races:
            random.shuffle(names[r][g])

    # Setup closed prompt
    closed_promptNo = "Vignette: " + closed.Vignette[0] + "\n" + \
        "Question: " + closed.Question[0] + "\n" + \
        "Answer: " + closed.Answer[0] + "\n" + \
        "Dosage: " + str(closed.Dosage[0]) + "\n" + \
        "Explanation: " + closed.Explanation[0] + "##\n\n"

    # Start experiment
    results_list = []
    call_count = 0
    total_calls = 80

    print(f"\n🚀 Starting experiment at {time.strftime('%H:%M:%S')}")
    print("=" * 80)

    start_time = time.time()

    for q in range(10):
        vignette_start = time.time()
        print(f"\n{'='*60}")
        print(f"VIGNETTE {q}/10")
        print(f"{'='*60}")

        # Sample closed prompts
        available = [i for i in range(10) if i != q]
        sample = random.sample(available, 2)
        closed_prompt0 = create_closed(sample[0], dose_high, data)
        closed_prompt2 = create_closed(sample[1], dose_low, data)

        closed_prompt0 = standardize_closed(closed_prompt0, 'Patient A')
        closed_prompt2 = standardize_closed(closed_prompt2, 'Patient C')
        # NOTE (verbatim): closed_prompt2 ('Patient C', Low) is built but NOT included,
        # exactly as in the original scripts.
        closed_prompt = closed_prompt0 + closed_promptNo

        if retr_evidence:
            # Retrieved evidence for open prompt from the new columns in the data
            similar_evidence = "\n\n<context> #Relevant Clinical Guideline Recommendations:\n\n" +data.iloc[q, 5] + "</context>\n\n"
            similar_mmr_evidence = "\n\n<context> #Relevant Clinical Guideline Recommendations:\n\n" +data.iloc[q, 6] + "</context>\n\n"

            relevant_guidelines = similar_evidence
        else:
            relevant_guidelines = ""

        # Generate for all demographics
        for g in genders:
            open_prompt_standard = create_open_standard(q, data)
            open_prompt_gendered = genderize_open(open_prompt_standard, g, pronouns)

            for r in races:
                open_prompt = relevant_guidelines + race_name_open(open_prompt_gendered, r, g, q, names)
                final_prompt = closed_prompt + open_prompt

                # API call with retry logic
                max_retries = 5
                response = None

                for attempt in range(max_retries):
                    try:
                        response = llm.chat.completions.create(
                            model=model_name,
                            messages=[{'role': 'user', 'content': final_prompt}],
                            max_tokens=MAX_TOKENS,
                            temperature=TEMPERATURE,
                            n=1,
                            logprobs=True,
                            top_logprobs=TOP_LOGPROBS
                        )
                        break
                    except Exception as e:
                        print(f"\n⚠️  Error on call {call_count+1} - Attempt {attempt+1}/{max_retries}")
                        print(f"   {r} {g}, Vignette {q}")
                        print(f"   {str(e)[:100]}")
                        if attempt < max_retries - 1:
                            sleep_time = 2 * (2 ** attempt)
                            print(f"   Retrying in {sleep_time} seconds...")
                            time.sleep(sleep_time)
                        else:
                            print(f"   ❌ Failed after all retries")
                            response = None

                if response is None:
                    print(f"   ⚠️  Skipping call {call_count+1}")
                    continue

                # Extract data
                choice = response.choices[0]
                gen_text = choice.message.content
                prob_yes, prob_no, prob_high, prob_low = extract_probabilities(choice)
                answer, dosage, explanation = parse_response_text(gen_text)

                # Store result
                results_list.append({
                    'prompt_num': q,
                    'race': r,
                    'gender': g,
                    'name': names[r][g][q],
                    'prob_yes': prob_yes,
                    'prob_no': prob_no,
                    'prob_high': prob_high,
                    'prob_low': prob_low,
                    'answer': answer,
                    'dosage': dosage,
                    'explanation': explanation,
                    'full_text': gen_text,
                    'closed_prompt': closed_prompt,
                    'open_prompt': open_prompt
                })

                call_count += 1

                # Progress tracking
                if call_count % 10 == 0:
                    elapsed = time.time() - start_time
                    calls_per_sec = call_count / elapsed
                    eta_seconds = (total_calls - call_count) / calls_per_sec
                    print(f"  ✓ Progress: {call_count}/{total_calls} ({call_count/total_calls*100:.0f}%) | ETA: {eta_seconds/60:.1f}min")

                # Auto-save backup every 40 calls
                if call_count % 40 == 0:
                    save_backup(results_list, call_count, backup_dir, backup_prefix)

                # 1-second sleep between calls
                time.sleep(SLEEP_BETWEEN_CALLS)

        vignette_time = time.time() - vignette_start
        print(f"✓ Vignette {q} complete: 8 calls in {vignette_time:.1f}s")

    # Save final results
    total_time = time.time() - start_time
    print(f"\n{'='*80}")
    print(f"🎉 EXPERIMENT COMPLETE!")
    print(f"{'='*80}")
    print(f"Total calls: {call_count}/{total_calls}")
    print(f"Total time: {total_time/60:.2f} minutes")
    if call_count:
        print(f"Average: {total_time/call_count:.2f} seconds per call")
    print("=" * 80)

    # Create final DataFrame
    final_df = to_dataframe(results_list)

    # Save
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    final_df.to_csv(output_file, index=False)
    print(f"\n✅ Results saved: {output_file}")
    print(f"   Total records: {len(final_df)}")
    print("=" * 80)
    return output_file


## 3. Run all models × retrieval × conditions

In [ ]:
overall_start = time.time()
outputs = []
for model_name, model_tag in MODELS.items():
    for retr_evidence in RETRIEVAL_VARIANTS:
        for condition in CONDITIONS_TO_RUN:
            outputs.append(run_experiment(model_name, model_tag, condition, retr_evidence))

print(f"\nAll runs finished in {(time.time() - overall_start)/60:.1f} min")
for o in outputs:
    print(" -", o)


## 4. Sanity check

In [ ]:
# ==== Quick check: share of Low decisions per system × condition (cf. Table 3 of the preprint) ====
import re

def classify_option(dosage):
    s = str(dosage).lower()
    for label, pat in [("None", r"none of the above|\bnone\b"), ("Medium", r"\bmedium\b"),
                       ("High", r"\bhigh\b"), ("Low", r"\blow\b")]:
        if re.search(pat, s):
            return label
    return "Unparsed"

rows = []
for f in sorted(glob.glob(os.path.join(OUTPUT_ROOT, "tier1_*", "*.csv"))):
    df = pd.read_csv(f)
    opts = df["gpt4o_dosage"].map(classify_option)
    rows.append({
        "condition": os.path.basename(os.path.dirname(f)),
        "file": os.path.basename(f),
        "n": len(df),
        **{f"%{k}": round(100 * (opts == k).mean(), 1) for k in ["Low", "Medium", "High", "None", "Unparsed"]},
    })
summary = pd.DataFrame(rows)
summary
